### **Create Bulk Analysis**

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from quinex_application.quinex import Quinex
quinex = Quinex()

In [2]:
name = "quenix_application_query2"
response = quinex.intialize_analysis(name)
print(response)

{'detail': 'Analysis with name quenix_application_query2 already exists. Please choose another name.'}


### **Import and Sort Paper by Year**

In [3]:
from quinex_application.visualisation_metadata import VisualisationMetadata

visualisation = VisualisationMetadata("../data/external/scopus_query2/scopus_2025_05_08_cleaned.csv")
df_doi = visualisation.df

In [4]:
# Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2025 = dfs_by_year[2025]
print(len(df_2025))

1379


### **Add Papers via DOI**

In [6]:
from quinex_application.mongos_store import MongoStore
mongodb = MongoStore(quinex, database_name="quinex_db_query2", collection_name="papers_2025")

In [ ]:
mongodb.insert_paper(name, df_2025)

In [9]:
collections = mongodb.get_collection("papers_2025")
print(len(collections))

577


### **Successfully Added Papers**

In [7]:
dois_df = mongodb.get_papers_from_collection(df_2025, "papers_2025")

### **Perform Bulk Analysis**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
#mongodb.process_paper(name, dois)

### **Add Bulk Analysis to MongoDB**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.add_analysis(name, dois)

In [ ]:
response = mongodb.add_analysis(name, ["10.1016/j.rineng.2025.104155"])

In [ ]:
def find_large_ints(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            yield from find_large_ints(v, new_path)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            new_path = f"{path}[{i}]"
            yield from find_large_ints(v, new_path)
    elif isinstance(obj, int):
        if abs(obj) > 2**63 - 1:
            yield path, obj

for path, value in find_large_ints(response):
    print(f"🚨 Zu große Zahl bei '{path}': {value}")

In [ ]:
print(response["annotations"]["quantitative_statements"][132])